In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfixest as pf

REF_YEAR = 2016          # last pre-treatment fall
YEARS = sorted(long["fall_year"].unique())


# ------------------------------------------------------------------
# 1. Visual check: treated vs comparison, one panel per group.
#    Log scale + indexed to 2016, so "parallel" = same slope, and groups of
#    very different sizes can be read on the same axis.
# ------------------------------------------------------------------
def plot_trends(data, group_col, fname, ncols=4):
    groups = sorted(data[group_col].unique())
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows),
                             sharex=True, sharey=True, squeeze=False)
    for ax, g in zip(axes.flat, groups):
        sub = data[data[group_col] == g]
        m = sub.groupby(["fall_year", "treated"])["y"].mean().unstack()
        m = m - m.loc[REF_YEAR]                        # index to 2016 = 0
        ax.plot(m.index, m[1], marker="o", label="Treated")
        ax.plot(m.index, m[0], marker="o", linestyle="--", label="Comparison")
        ax.axvline(REF_YEAR + 0.5, color="grey", linestyle=":")
        ax.axhline(0, color="lightgrey", linewidth=0.8)
        ax.set_title(g, fontsize=10)
    for ax in axes.flat[len(groups):]:
        ax.set_visible(False)
    axes.flat[0].legend(fontsize=8)
    fig.supxlabel("Fall year")
    fig.supylabel("Mean log(1 + FTFT enrollment), relative to 2016")
    fig.tight_layout()
    fig.savefig(figure_dir / fname, dpi=300, bbox_inches="tight")   # save AFTER labels
    plt.show()


plot_trends(race_long, "race", "pretrends_by_race.png")
plot_trends(rxg_long, "cell", "pretrends_by_race_gender.png")


# ------------------------------------------------------------------
# 2. Formal check: event study per group.
#    y = sum_{k != 2016} b_k * treated * 1[year = k] + unit FE + year FE
#    Pre-period b_k (k < 2016) should be ~0; joint F-test on them.
# ------------------------------------------------------------------
def event_study(data):
    d = data.copy()
    terms = []
    for k in YEARS:
        if k == REF_YEAR:
            continue
        name = f"ev_{k}"
        d[name] = d["treated"] * (d["fall_year"] == k)
        terms.append(name)
    fit = pf.feols(f"y ~ {' + '.join(terms)} | unitid + fall_year",
                   data=d, vcov={"CRV1": "unitid"})
    return fit


def pretrend_table(data, group_col):
    rows, coefs = [], []
    for g in sorted(data[group_col].unique()):
        fit = event_study(data[data[group_col] == g])
        c, se = fit.coef(), fit.se()
        names = list(c.index)
        pre = [n for n in names if int(n.split("_")[1]) < REF_YEAR]
        R = np.zeros((len(pre), len(names)))
        for i, n in enumerate(pre):
            R[i, names.index(n)] = 1
        w = fit.wald_test(R=R)
        rows.append({"Group": g,
                     "Pre-period F": float(np.ravel(w["statistic"])[0]),
                     "Pre-period p": float(np.ravel(w["pvalue"])[0])})
        for n in names:
            coefs.append({"Group": g, "Year": int(n.split("_")[1]),
                          "coef": c[n], "se": se[n]})
    return pd.DataFrame(rows), pd.DataFrame(coefs)


def plot_event_study(coefs, fname, ncols=4):
    groups = coefs["Group"].unique()
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows),
                             sharex=True, squeeze=False)
    for ax, g in zip(axes.flat, groups):
        s = coefs[coefs["Group"] == g]
        s = pd.concat([s, pd.DataFrame([{"Group": g, "Year": REF_YEAR,
                                         "coef": 0, "se": 0}])]).sort_values("Year")
        ax.errorbar(s["Year"], s["coef"], yerr=1.96 * s["se"], fmt="o-", capsize=3)
        ax.axhline(0, color="grey", linewidth=0.8)
        ax.axvline(REF_YEAR + 0.5, color="red", linestyle="--")
        ax.set_title(g, fontsize=10)
    for ax in axes.flat[len(groups):]:
        ax.set_visible(False)
    fig.supxlabel("Fall year")
    fig.supylabel("Treated - comparison, log points (2016 = 0)")
    fig.tight_layout()
    fig.savefig(figure_dir / fname, dpi=300, bbox_inches="tight")
    plt.show()


race_pre, race_es = pretrend_table(race_long, "race")
display(race_pre.round(3))
plot_event_study(race_es, "event_study_by_race.png")

cell_pre, cell_es = pretrend_table(rxg_long, "cell")
display(cell_pre.round(3))
plot_event_study(cell_es, "event_study_by_race_gender.png")

NameError: name 'long' is not defined